# What a four-point probe measures
Drive a current between two outer point contacts. Measure the open-circuit voltage between the equal-spaced inner contacts. Predict whether doubling all spacings changes the measured resistance before comparing the thin-film and bulk limits.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rho_ohm_m = 2.0e-5
thickness_um = 20.
spacing_um = 80.
current_mA = 5.
assert rho_ohm_m > 0 and thickness_um > 0 and current_mA>0
s_over_h = spacing_um/thickness_um
assert .04 <= s_over_h <= 15, "Use spacing/thickness between 0.04 and 15"

## Sum the image sources before subtracting large potentials
In units of the thickness, a pair of mirror distances is a and b. Compute 1/a − 1/b by rationalising it. The resulting terms decay as the inverse cube of image index.

In [ ]:
def scaled_resistance(z, images=6000):
    z=np.atleast_1d(np.asarray(z,dtype=float))
    n=2*np.arange(1,images+1,dtype=float)
    a=np.hypot(z[:,None],n); b=np.hypot(2*z[:,None],n)
    diff=3*z[:,None]**2/(a*b*(a+b))
    return (1/(2*z)+2*diff.sum(axis=1))/np.pi

sheet=rho_ohm_m/(thickness_um*1e-6)
ratio=scaled_resistance(s_over_h)[0]
voltage_mV=current_mA*sheet*ratio
print(f"Sheet resistance {sheet:.6g} ohm per square, voltage {voltage_mV:.6g} mV.")
print(f"Apparent sheet resistance from thin-film formula {np.pi*voltage_mV/(current_mA*np.log(2)):.6g} ohm per square.")

## View the current geometry and the thickness crossover
The left panel shows equipotentials of the ideal sheet geometry in the plane. The right panel solves the finite-thickness image problem and compares two asymptotes. The point contacts themselves lie outside the model's finite potential range.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(9,3.9),layout="constrained")
x=np.linspace(-2.6,2.6,190); y=np.linspace(-1.55,1.55,130); X,Y=np.meshgrid(x,y)
phi=np.log(np.hypot(X-1.5,Y)/np.hypot(X+1.5,Y))
axes[0].contour(X,Y,phi,levels=[-2,-1.2,-.7,-.35,0,.35,.7,1.2,2],colors="#175868",linewidths=1.1)
for pos,label,colour in zip([-1.5,-.5,.5,1.5],["+I","V₂","V₃","−I"],["#a23d21","#375b9a","#375b9a","#a23d21"]):
    axes[0].plot(pos,0,"o",color=colour);axes[0].text(pos,.17,label,ha="center",bbox=dict(fc="white",ec="none",pad=1))
axes[0].set(xlabel="Along probes  x/s",ylabel="Across probes  y/s",aspect="equal",title="Infinite-sheet equipotentials")
z=np.geomspace(.08,15,180)
axes[1].loglog(z,scaled_resistance(z),label="Finite thickness",color="#175868",lw=2.4)
axes[1].loglog(z,1/(2*np.pi*z),"--",label="Bulk",color="#375b9a")
axes[1].axhline(np.log(2)/np.pi,ls=":",label="Thin sheet",color="#a23d21")
axes[1].plot(s_over_h,ratio,"o",color="#a23d21");axes[1].legend(fontsize=9)
axes[1].set(xlabel="Spacing / thickness  s/h",ylabel="Scaled resistance  (V/I)h/ρ",title="What the voltmeter measures")
plt.show()

## Challenge the approximations
Increase the image cutoff and compare the numerical uncertainty with the thin-sheet bias. At small spacing the voltage drop decreases approximately as 1/s. Explain why this also changes how a fixed voltmeter uncertainty propagates into a resistivity estimate.

In [ ]:
value2=scaled_resistance(s_over_h,12000)[0]
assert abs(value2-ratio) < 5e-6
assert np.isclose(scaled_resistance(.005)[0],1/(2*np.pi*.005),rtol=1e-5)
assert abs(scaled_resistance(12)[0]-np.log(2)/np.pi) < 2e-6
print("Fractional thin-sheet bias:",np.pi*ratio/np.log(2)-1)

## Optional OneSimulation fit with a thermoelectric offset
A constant voltage offset should enter as a separate fit coefficient. The positive and negative current readings below share the same physical resistance and offset. The synthetic voltages have no random error, so an independent linear solve must recover the input parameters.

In [ ]:
try:
    from onesim.numerics.linear.matrices.dense_linear_algebra import least_squares_dense
except ImportError:
    print("The optional sensor fit runs in a local OneSimulation installation.")
else:
    spacings=np.array([.2,.5,1.,2.])
    k=np.repeat(scaled_resistance(spacings),2)*np.tile([current_mA,-current_mA],len(spacings))
    design=np.column_stack([k,np.ones(len(k))])
    readings=k*sheet+0.012
    recovered=least_squares_dense(design,readings)
    print("Estimated sheet resistance in ohms and offset in mV:",recovered)
    assert np.allclose(recovered,[sheet,.012])